# Image edits and source-image workflows

Use `gpt-image-2` for source-image edits, including inpainting and product placement. These calls send image bytes to Azure OpenAI and may incur generation costs. Configure auth, then set `RUN_IMAGE_EDITS=1` to execute paid cells.

In [ ]:
from pathlib import Path
import os
import sys

from IPython.display import Image, display
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "packages" / "t2i_core").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "packages" / "t2i_core" / "src"))
load_dotenv(PROJECT_ROOT / ".env")

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "image_edits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_IMAGE_EDITS = os.getenv("RUN_IMAGE_EDITS") == "1"
print("Paid edit calls enabled:", RUN_IMAGE_EDITS)

In [ ]:
from t2i_core import GPTImageProvider, Settings
from t2i_core.scenarios import inpaint_image, place_product

settings = Settings()
provider = GPTImageProvider(settings)
print("Editing deployment:", provider.deployment)

## Inpainting with a source image and mask

The mask must be a PNG with an alpha channel and the same dimensions as the source image. Transparent pixels identify the editable area.

In [ ]:
source_path = PROJECT_ROOT / "app" / "sample_assets" / "inpainting" / "example-1" / "source.png"
mask_path = PROJECT_ROOT / "app" / "sample_assets" / "inpainting" / "example-1" / "mask.png"
display(Image(filename=str(source_path)))
display(Image(filename=str(mask_path)))

edit_prompt = (
    "Replace the masked area with a small brushed stainless steel espresso cup, "
    "matching the source lighting, perspective, shadows, and photographic style. "
    "Preserve all unmasked parts of the original image."
)

if RUN_IMAGE_EDITS:
    results = await inpaint_image(
        provider,
        source_path.read_bytes(),
        edit_prompt,
        mask=mask_path.read_bytes(),
        size="1024x1024",
        quality="high",
    )
    output_path = OUTPUT_DIR / "inpainting-result.png"
    output_path.write_bytes(results[0].image)
    print(results[0].model, results[0].usage.model_dump())
    display(Image(filename=str(output_path)))
else:
    print("Set RUN_IMAGE_EDITS=1 to run the inpainting call.")

## Product placement

Product placement reuses one product source image across one or more environment prompts. Keep prompts explicit about what to preserve from the product image.

In [ ]:
product_path = PROJECT_ROOT / "app" / "sample_assets" / "product-placement" / "example-1" / "product.png"
display(Image(filename=str(product_path)))

environments = [
    "a bright Scandinavian kitchen counter with soft window light; preserve the product label and shape",
    "a premium outdoor picnic scene on a linen cloth at golden hour; preserve product proportions",
]

if RUN_IMAGE_EDITS:
    results = await place_product(provider, product_path.read_bytes(), environments, size="1024x1024", quality="high")
    for index, result in enumerate(results, start=1):
        output_path = OUTPUT_DIR / f"product-placement-{index}.png"
        output_path.write_bytes(result.image)
        print(index, result.model, result.usage.model_dump())
        display(Image(filename=str(output_path)))
else:
    print("Set RUN_IMAGE_EDITS=1 to run product-placement edits.")

await provider.client.close()